# Lot D — EXP-D-013 : Falcon-H1-7B contre Aya Expanse 8B

**Auteur : Membre 5 (Hadil).**

**Plan fixé avant la mesure (ce notebook est commité avant toute exécution) :**
- modèle : `tiiuae/Falcon-H1-7B-Instruct` (7,6 Md ≤ 8 Md ; 18 langues dont l'arabe et le français ;
  meilleur modèle ≤ 8 Md sur les dialectes arabes dans DialectalArabicMMLU, qui n'a pas de tunisien) ;
- même protocole qu'EXP-D-010 : `rag_llm.silver_verdict --main-only`, prompt v1, température 0.3,
  top-p 0.9, 4 bits sur GPU T4, recherche hybride top 5, NLI Bronze pour le gabarit de repli,
  graines 13 / 42 / 2026 ; dev d'abord (sorties valides), puis test gelé **une seule fois** ;
- comparaison appariée avec Aya Expanse 8B (EXP-D-010) sur la graine 42 : `rag_llm.compare_llm_runs`
  (bootstrap 10 000 tirages, exactitude et macro-F1), sur le PC ;
- **règle de décision** : Falcon-H1-7B remplace Aya 8B comme système principal si (1) ses sorties
  valides sur le test sont ≥ 0,90 et (2) son exactitude moyenne sur 3 graines dépasse celle d'Aya 8B
  (0,730). Sinon Aya 8B reste. Avec 21 affirmations, l'écart et son IC sont toujours rapportés, et un
  écart non significatif est dit tel quel. Le tunisien (5 affirmations) est rapporté à part, sans
  servir de critère (trop petit).

**Correctif technique (18/11, avant toute mesure) :** le premier essai s'est arrêté sur une erreur `inf` / `nan` (calcul en float16, le GPU T4 n'a pas de bfloat16). Le calcul se fait en **float32** (`LLM_COMPUTE_DTYPE`) ; aucun résultat n'avait été produit, et le reste du protocole ne change pas.

Avant de commencer : *Exécution → Modifier le type d'exécution → **GPU T4***.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Jetons (cases masquées) et projet (branche `lot-d-falcon`)

In [ ]:
import os, subprocess, base64, sys
from pathlib import Path
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Jeton Hugging Face Read : ").strip()
token = getpass("Jeton GitHub (lecture du dépôt) : ").strip()
auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
env = dict(os.environ, GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
           GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + auth, GIT_TERMINAL_PROMPT="0")
if not Path("/content/NewsCheckTN/.git").exists():
    r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-falcon",
                        "https://github.com/SouissiOumaima/NewsCheckTN.git", "/content/NewsCheckTN"],
                       env=env, capture_output=True)
    if r.returncode:
        raise RuntimeError("Clone privé échoué : vérifier le jeton et le nom de la branche")
r = subprocess.run(["git", "-C", "/content/NewsCheckTN", "fetch", "-q", "origin", "lot-a1-data"],
                   env=env, capture_output=True)
del token, auth, env
os.chdir("/content/NewsCheckTN")
print("✅ commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(),
      "| lot-a1-data :", "OK" if r.returncode == 0 else "échec")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt",
                "sentence-transformers", "bitsandbytes", "accelerate", "-U", "transformers"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
import transformers
print("✅ dépendances installées | transformers", transformers.__version__)

## 2. Corpus de test (privé, reconstruit dans la session, jamais poussé)

In [ ]:
!python -m rag_llm.build_tunifact_eval
!python -m rag_llm.freeze_test --check

## 3. Configuration (identique à EXP-D-010, seul le modèle change)

In [ ]:
os.environ.update({
    "LLM_PROVIDER": "hf",
    "LLM_MODEL": "tiiuae/Falcon-H1-7B-Instruct",
    "RETRIEVAL_METHOD": "hybrid",
    "NLI_BACKEND": "bronze",
    "LLM_COMPUTE_DTYPE": "float32",  # float16 : inf / nan sur T4
})
print({k: os.environ[k] for k in ("LLM_PROVIDER", "LLM_MODEL", "RETRIEVAL_METHOD", "NLI_BACKEND", "LLM_COMPUTE_DTYPE")})

## 4. Essai rapide (3 affirmations du dev)
Le premier lancement télécharge le modèle (~15 Go). Regarder : sorties valides, et le temps total.
Un avertissement « fast path is not available » (noyaux Mamba absents) est normal : c'est seulement plus lent.

In [ ]:
import time
t = time.time()
!python -X utf8 -m rag_llm.silver_verdict --main-only --limit 3
print(f"Temps : {(time.time() - t) / 60:.1f} min")

## 5. Dev complet (20 affirmations, 3 graines)
**Envoyer ce résultat à Claude avant le test.**

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits dev

## 6. Test gelé (21 affirmations) — une seule fois

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits test

## 7. Récupérer les résultats
Contient des extraits Tunifact : **privé**, à garder sur le PC (jamais sur GitHub).

In [ ]:
!cd evaluation/results && zip -q ../../lot-d_exp-d-013_falcon.zip lot-d_silver_verdict_*Falcon*.json lot-d_llm_cache.jsonl
from google.colab import files
files.download("lot-d_exp-d-013_falcon.zip")